In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Cell 3 – Imports & Config (Code)**

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

import matplotlib.pyplot as plt

try:
    import keras_tuner as kt
except ImportError as e:
    print("KerasTuner not installed:", e)

print("TensorFlow version:", tf.__version__)


**Cell 4 – Dataset Path & Class Names (Code)**

In [ ]:
# Update this if your dataset path is different
DATA_ROOT = "/kaggle/input/colon-histopathology-9-class"  # example

CLASS_NAMES = [
    "Normal", "Tumor", "Stroma", "Lympho",
    "Complex", "Debris", "Mucosa", "Adipose", "Background"
]

IMG_SIZE = (224, 224)
NUM_CLASSES = len(CLASS_NAMES)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
SEED = 42

os.listdir(DATA_ROOT), CLASS_NAMES


**Cell 5 – Data Loader & Preprocessing (Code)**

In [ ]:
def load_image_paths_and_labels(root_dir, class_names):
    image_paths, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(root_dir, cls)
        for fname in os.listdir(cls_dir):
            if fname.lower().endswith((".png", ".jpg", ".jpeg", ".tif")):
                image_paths.append(os.path.join(cls_dir, fname))
                labels.append(idx)
    return np.array(image_paths), np.array(labels)


def decode_image(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, IMG_SIZE)

    if augment:
        img = augment_image(img)

    img = (img - tf.constant([0.485, 0.456, 0.406])) / tf.constant(
        [0.229, 0.224, 0.225]
    )
    return img, label


def tfa_rotate(img, angle_rad):
    try:
        import tensorflow_addons as tfa
        img = tfa.image.rotate(img, angle_rad)
    except ImportError:
        pass
    return img


def augment_image(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    angle = tf.random.uniform([], -20.0, 20.0) * np.pi / 180.0
    img = tfa_rotate(img, angle)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    return img


def build_tf_dataset(paths, labels, batch_size, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(lambda p, l: decode_image(p, l, augment=augment),
                num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds


paths, labels = load_image_paths_and_labels(DATA_ROOT, CLASS_NAMES)
print("Total samples:", len(paths))


**Cell 6 – Splits (Code)**

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

train_ds = build_tf_dataset(X_train, y_train, BATCH_SIZE, augment=True, shuffle=True)
val_ds   = build_tf_dataset(X_val,   y_val,   BATCH_SIZE, augment=False, shuffle=False)
test_ds  = build_tf_dataset(X_test,  y_test,  BATCH_SIZE, augment=False, shuffle=False)


**Cell 7 – Macro-F1 Metric (Code)**

In [ ]:
class MacroF1Metric(keras.metrics.Metric):
    def __init__(self, num_classes, name="macro_f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.conf_matrix = self.add_weight(
            "conf_matrix",
            shape=(num_classes, num_classes),
            initializer="zeros",
            dtype=tf.float32
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        cm = tf.math.confusion_matrix(
            y_true, y_pred, num_classes=self.num_classes, dtype=tf.float32
        )
        self.conf_matrix.assign_add(cm)

    def result(self):
        cm = self.conf_matrix
        tp = tf.linalg.diag_part(cm)
        predicted = tf.reduce_sum(cm, axis=0)
        actual = tf.reduce_sum(cm, axis=1)

        precision = tf.math.divide_no_nan(tp, predicted)
        recall = tf.math.divide_no_nan(tp, actual)
        f1 = tf.math.divide_no_nan(2 * precision * recall, precision + recall)
        return tf.reduce_mean(f1)

    def reset_states(self):
        self.conf_matrix.assign(tf.zeros_like(self.conf_matrix))


**Cell 8 – EfficientNetB3 Model + BO Builder (Code)**

In [ ]:
def build_backbone_efficientnetb3(input_shape=IMG_SIZE + (3,), trainable_blocks=3):
    from tensorflow.keras.applications import EfficientNetB3
    base_model = EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape
    )
    for layer in base_model.layers:
        layer.trainable = False
    for layer in base_model.layers[-trainable_blocks * 20:]:
        layer.trainable = True
    return base_model


def build_classifier_head(x, hp=None):
    x = layers.GlobalAveragePooling2D()(x)
    if hp is not None:
        dense_units = hp.Choice("dense_units", [256, 512, 1024])
        dropout_rate = hp.Float("dropout_rate", 0.2, 0.6, step=0.1)
    else:
        dense_units = 512
        dropout_rate = 0.4
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return outputs


def model_builder(hp):
    base_model = build_backbone_efficientnetb3()
    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = base_model(inputs, training=False)
    outputs = build_classifier_head(x, hp=hp)
    model = keras.Model(inputs, outputs)

    lr = hp.Float("learning_rate", 1e-5, 1e-3, sampling="log")
    weight_decay = hp.Float("weight_decay", 1e-6, 1e-3, sampling="log")

    try:
        from tensorflow_addons.optimizers import AdamW
        optimizer = AdamW(learning_rate=lr, weight_decay=weight_decay)
    except ImportError:
        optimizer = keras.optimizers.Adam(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy",
            MacroF1Metric(NUM_CLASSES),
            keras.metrics.AUC(name="auc")
        ],
    )
    return model


**Cell 9 – Bayesian Optimization (Code)**

In [ ]:
import keras_tuner as kt

tuner = kt.BayesianOptimization(
    hypermodel=model_builder,
    objective=kt.Objective("val_macro_f1", direction="max"),
    max_trials=20,         # you can increase to 50 as in paper
    executions_per_trial=1,
    directory="kt_logs",
    project_name="colon_effnetb3_bo",
    overwrite=True
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_macro_f1",
    patience=10,
    mode="max",
    restore_best_weights=True
)

tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

best_hp = tuner.get_best_hyperparameters(1)[0]
best_hp.values


**Cell 10 – Train Best Model & Evaluate (Code)**

In [ ]:
best_model = model_builder(best_hp)

history = best_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop],
    verbose=1
)

# Evaluation on test set
y_true_all, y_pred_all, y_prob_all = [], [], []

for x_batch, y_batch in test_ds:
    probs = best_model.predict(x_batch, verbose=0)
    y_prob_all.append(probs)
    y_pred_all.append(np.argmax(probs, axis=1))
    y_true_all.append(y_batch.numpy())

y_true = np.concatenate(y_true_all)
y_pred = np.concatenate(y_pred_all)
y_prob = np.concatenate(y_prob_all)

acc = np.mean(y_true == y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
auc_macro = roc_auc_score(y_true_bin, y_prob, average="macro", multi_class="ovr")

print("Test Accuracy:", acc)
print("Test Macro-F1:", macro_f1)
print("Test Macro-AUC:", auc_macro)
print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


3️⃣ Complexity–Performance Plots (FLOPs vs Accuracy)

In [ ]:
import matplotlib.pyplot as plt

# Example values – replace with your real numbers
models = ["VGG19 + BO", "EffNetB3 + BO"]
params_m = [20.3, 14.4]
flops_b = [19.6, 12.0]
accuracy = [94.8, 97.1]

# FLOPs vs Accuracy
plt.figure()
for m, f, a in zip(models, flops_b, accuracy):
    plt.scatter(f, a)
    plt.text(f + 0.1, a, m)
plt.xlabel("FLOPs (Billions)")
plt.ylabel("Accuracy (%)")
plt.title("Complexity–Performance Trade-off (FLOPs vs Accuracy)")
plt.grid(True)
plt.show()

# Params vs Accuracy
plt.figure()
for m, p, a in zip(models, params_m, accuracy):
    plt.scatter(p, a)
    plt.text(p + 0.1, a, m)
plt.xlabel("Parameters (Millions)")
plt.ylabel("Accuracy (%)")
plt.title("Complexity–Performance Trade-off (Params vs Accuracy)")
plt.grid(True)
plt.show()
